In [ ]:
import sys, os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/joshuaswords/netflix-data-visualization/src/small_bench/checkpoints/pre_cell_17.pickle

In [ ]:
%%PandasProfile
### cell 17 ###

# 1) Compute df_polar by adding 'Value' and reversing order via sort_index
#    (avoids reset_index() + sort_values() on a column)
df_polar = (
    data_sub
    .assign(Value=lambda x: x['Movie'] + x['TV Show'])
    .sort_index(ascending=False)
    .reset_index()
)

# 2) Build color_map in one expression (no Python‐level loop or post‐mutation)
n = len(df_polar)
if n > 1:
    color_map = ['#b20710'] + ['#221f1f'] * (n - 2) + ['#b20710']
else:
    color_map = ['#b20710'] * n

# 3) Constants
lowerLimit, labelPadding = 1, 30

# 4) Vectorized height calculation (no explicit slope variable reuse)
vals = df_polar['Value']
max_val = vals.max()
heights = (max_val - lowerLimit) * vals / max_val + lowerLimit

# 5) Compute bar width and angles via NumPy
width = 2 * np.pi / n
angles = np.linspace(width, 2 * np.pi, n)

angles